*— cell 0 —*

# T04 Arm 2A (metadata) — Azure GPU precompute, remaining 4 stems

Re-precompute the dense + sparse indices for **doc 5, 6, 7, 9** under
`LINKER_VERSION = 4` on an Azure ML compute instance with a CUDA GPU.
Doc 8 is already done (built earlier on 2026-05-26 under v4).

**Properties:**
- Per-stem cells let you start/stop/monitor each PDF independently.
- Cell-level idempotency: each stem cell skips entirely if its 6
  SMOKE_PLAN configs are already on disk under linker v4 (no script
  invocation, no model load). Safe to re-run cells after kernel
  restarts.
- The script (`precompute_t04_indices.py`) is also idempotent at the
  (unit, variant) granularity — interrupting a stem mid-build and
  re-running its cell picks up at the next un-built variant.

**Sequence:** GPU sanity → clone repos → pip deps → download bundle →
preflight → **smoke (= full doc 5 build, measures per-variant times)**
→ time gate (extrapolates the other 3 stems) → per-stem cells (4) →
verify → upload → stage for `scp` back → cleanup.


In [1]:
# === cell 1 ===
# Edit these for the run.
GITHUB_TOKEN = ""                                # PAT with read scope on the 5 sibling repos
GITHUB_OWNER = "MariusPasch"
MONO_REPO = "bsard-rag-thesis"
GITHUB_BRANCH = "main"

# Five sibling repos required at runtime (T01–T04 + T07).
SIBLING_REPOS = {
    "T01_SHARED":      "RQ2_T01_SHARED",
    "T02_DATA_LOADER": "RQ2_T02_DATA_LOADER",
    "T03_ARM1_NAIVE":  "RQ2_T03_ARM1_NAIVE",
    "T04_ARM2_METADATA": "RQ2_T04_ARM2_METADATA",
    "T07_EVALUATION":  "RQ2_T07_EVALUATION",
}

# Container-level SAS URL with read+write+list. Bundle must already exist.
AZURE_CONTAINER_SAS_URL = ""  # paste your container SAS URL (read+write+list) before running
BUNDLE_BLOB_NAME = "t04_azure_bundles/v4_remaining4.zip"
RESULTS_BLOB_PREFIX = "t04_results/v4_remaining4"

# Paths on the Azure compute instance.
REPOS_DIR = "/home/azureuser/repos"
BUNDLE_DIR = "/home/azureuser/bundle"
RESULTS_DIR = "/home/azureuser/results"

# Stems to process. Doc 5 is smallest → used as the smoke target.
# After the smoke cell doc 5 is fully built, and the doc 5 per-stem cell
# below will skip via the idempotency check.
STEMS = [
    "1967_10_10_1967101056",   # doc 5 — Code Judiciaire (smaller)  ← smoke target
    "1867_06_08_1867060850",   # doc 6 — Code Pénal
    "1804_03_21_1804032150",   # doc 9 — Code Civil
    "1967_10_10_1967101055",   # doc 7 — Code Judiciaire (larger)
]
SMOKE_STEM = STEMS[0]

# SMOKE_PLAN — must match arm2_metadata SMOKE_PLAN exactly (used for
# both the smoke and the completion check). Two units × variants split:
#   4 node variants + 2 article variants = 6 configs per stem.
SMOKE_PLAN = [
    ("node",    "raw"),
    ("node",    "enriched"),
    ("node",    "summary"),
    ("node",    "full"),
    ("article", "raw"),
    ("article", "full"),
]

# HF model id (must match the precompute config — part of compute_config_hash).
EMBEDDING_MODEL = "intfloat/multilingual-e5-large-instruct"

# Workload per stem (indexable nodes, indexable articles). Used by the
# time-gate cell to scale per-variant per-unit times. Source: AzureDI
# dump post-2026-05-20 corpus (see CHANGE_NOTES.md, 2026-05-25/27 entries).
WORKLOAD = {
    "1967_10_10_1967101056": ( 619,  294),   # doc 5
    "1867_06_08_1867060850": ( 680,  380),   # doc 6
    "1804_03_21_1804032150": ( 958,  422),   # doc 9
    "1967_10_10_1967101055": (1616,  762),   # doc 7
}

# Where smoke per-variant timings are persisted (so the time-gate cell
# survives kernel restarts and so a re-run of the smoke cell on cache-hit
# data falls back to the original measurements).
SMOKE_TIMINGS_FILE = ".smoke_timings.json"


*— cell 2 —*

## 1. GPU sanity check

Bail early if there is no CUDA device — the whole point of moving to Azure
is to use the GPU. The script would still run on CPU but at the local CPU
rate (~67 h for the 4 stems).

In [2]:
# === cell 3 ===
import subprocess
import sys

r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr)
if r.returncode != 0:
    raise RuntimeError("nvidia-smi failed — no GPU on this compute instance.")

import importlib
try:
    torch = importlib.import_module("torch")
    print(f"torch:                {torch.__version__}")
    print(f"torch.cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"torch.cuda.device(0): {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING — torch sees no CUDA. Re-install with the CUDA wheel "
              "via cell 5 (pip install). Re-run this cell after install.")
except ImportError:
    print("torch not installed yet — that's fine, cell 5 will install it.")


Wed May 27 10:14:51 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       On  | 00000001:00:00.0 Off |                  Off |
| N/A   27C    P8               9W /  70W |      5MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

*— cell 4 —*

## 2. Clone the 5 sibling repos

Pulls T01–T04 + T07 from GitHub. Validates that any pre-existing dir's
`origin` matches the configured URL, refreshes the embedded PAT, and
hard-asserts `LINKER_VERSION = 4` is present in T04 before continuing.

In [3]:
# === cell 5 ===
import subprocess
from pathlib import Path

# The former sibling repos (T01-T04 + T07) are now subfolders of the single
# mono-repo, under RQ2_Structure_Aware_Retrieval/. Clone the mono-repo once.
MONO_DIR = Path(REPOS_DIR) / MONO_REPO
auth_url = (
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_OWNER}/{MONO_REPO}.git"
    if GITHUB_TOKEN else f"https://github.com/{GITHUB_OWNER}/{MONO_REPO}.git"
)
Path(REPOS_DIR).mkdir(parents=True, exist_ok=True)
if MONO_DIR.exists():
    subprocess.run(["git", "-C", str(MONO_DIR), "remote", "set-url", "origin", auth_url], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "fetch", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "pull", "--quiet"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", GITHUB_BRANCH, auth_url, str(MONO_DIR)], check=True)

# Each former sibling repo is now a subfolder of the mono-repo.
RQ2_ROOT = MONO_DIR / "RQ2_Structure_Aware_Retrieval"
repo_paths = {label: RQ2_ROOT / repo_name for label, repo_name in SIBLING_REPOS.items()}
for label, path in repo_paths.items():
    print(f"  {label:<18}  {path}")

ret_path = repo_paths["T04_ARM2_METADATA"] / "src" / "arm2_metadata" / "retriever.py"
content = ret_path.read_text(encoding="utf-8")
if "LINKER_VERSION = 4" not in content:
    raise RuntimeError(
        f"{ret_path} does NOT contain LINKER_VERSION = 4. Stop and fix "
        "before continuing — running under v3 would build useless caches."
    )
print("\nLINKER_VERSION = 4 verified in T04.")


Already on 'master'


Your branch is up to date with 'origin/master'.
  T01_SHARED          /home/azureuser/repos/RQ2_T01_SHARED  (3320877 doc 8 reprecompute)


Already on 'master'


Your branch is up to date with 'origin/master'.
  T02_DATA_LOADER     /home/azureuser/repos/RQ2_T02_DATA_LOADER  (ebc09e2 before t04 precompute)


Already on 'master'


Your branch is up to date with 'origin/master'.
  T03_ARM1_NAIVE      /home/azureuser/repos/RQ2_T03_ARM1_NAIVE  (8b1bf4f selected pdfs results analysis)


Already on 'master'


Your branch is up to date with 'origin/master'.
  T04_ARM2_METADATA   /home/azureuser/repos/RQ2_T04_ARM2_METADATA  (77769ab before azure precompute selected pdfs)


Already on 'master'


Your branch is up to date with 'origin/master'.
  T07_EVALUATION      /home/azureuser/repos/RQ2_T07_EVALUATION  (b72caf6 doc 8 reprecompute)

LINKER_VERSION = 4 verified in T04.


*— cell 6 —*

## 3. Install Python packages

CUDA-enabled PyTorch via the cu121 extra-index-url, then the standard
T04 dependencies, then editable installs of the 5 sibling packages.

In [4]:
# === cell 7 ===
import subprocess
import sys

PIP = [sys.executable, "-m", "pip", "install", "-q"]

import importlib
need_cuda_torch = True
try:
    torch = importlib.import_module("torch")
    if torch.cuda.is_available():
        need_cuda_torch = False
        print(f"  CUDA torch already installed ({torch.__version__})")
except ImportError:
    pass

if need_cuda_torch:
    print("  installing CUDA torch from cu121 wheel index...")
    subprocess.run(
        PIP + ["--extra-index-url", "https://download.pytorch.org/whl/cu121",
               "torch>=2.0"],
        check=True,
    )

deps = [
    "numpy>=1.26", "pandas>=2.0", "tqdm>=4.66",
    "sentence-transformers>=2.7", "transformers>=4.40",
    "faiss-cpu>=1.8", "rank-bm25>=0.2",
    "PyMuPDF",
    "azure-storage-blob",
]
subprocess.run(PIP + deps, check=True)

INSTALL_ORDER = ["T01_SHARED", "T02_DATA_LOADER", "T03_ARM1_NAIVE",
                 "T04_ARM2_METADATA", "T07_EVALUATION"]
for label in INSTALL_ORDER:
    target = repo_paths[label]
    subprocess.run(PIP + ["-e", str(target)], check=True)
    print(f"  installed {label} from {target}")

for label, target in repo_paths.items():
    src_path = str(target / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

from arm2_metadata.retriever import LINKER_VERSION
assert LINKER_VERSION == 4, f"runtime LINKER_VERSION = {LINKER_VERSION}, expected 4"
print(f"\nAll packages installed; arm2_metadata.LINKER_VERSION = {LINKER_VERSION}")


  CUDA torch already installed (2.9.1+cu128)
  installed T01_SHARED from /home/azureuser/repos/RQ2_T01_SHARED
  installed T02_DATA_LOADER from /home/azureuser/repos/RQ2_T02_DATA_LOADER
  installed T03_ARM1_NAIVE from /home/azureuser/repos/RQ2_T03_ARM1_NAIVE
  installed T04_ARM2_METADATA from /home/azureuser/repos/RQ2_T04_ARM2_METADATA
  installed T07_EVALUATION from /home/azureuser/repos/RQ2_T07_EVALUATION

All packages installed; arm2_metadata.LINKER_VERSION = 4


*— cell 8 —*

## 4. Download the input bundle from Azure Blob

Built locally with `scripts/prepare_azure_bundle.py` and uploaded once.
Idempotent — skipped if `bundle.zip` is already on disk.

**Bundle contents (~430 MB):** `azuredi/` (308 MB JSON + small CSVs),
`dataset_creation_output/bsard_corpus.db`, `pdf_document_map.csv`, and
the 4 source PDFs (needed for the pdf_sha256 fingerprint).

In [5]:
# === cell 9 ===
import zipfile
from pathlib import Path

from azure.storage.blob import ContainerClient

bundle_dir = Path(BUNDLE_DIR)
bundle_dir.mkdir(parents=True, exist_ok=True)

container = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)

# Derive the local filename from the blob path so a stale bundle.zip
# from a sibling notebook (e.g. T05) sharing this BUNDLE_DIR can't
# shadow our download. T05's previous run left a 141 KB bundle.zip
# here that made `if not local_zip.exists()` short-circuit and skip
# the T04 download entirely.
local_zip = bundle_dir / Path(BUNDLE_BLOB_NAME).name

if not local_zip.exists():
    print(f"Downloading {BUNDLE_BLOB_NAME} -> {local_zip.name} ...")
    with local_zip.open("wb") as f:
        f.write(
            container.get_blob_client(BUNDLE_BLOB_NAME)
            .download_blob().readall()
        )
size_mb = local_zip.stat().st_size / 1024 / 1024
print(f"Bundle zip: {local_zip} ({size_mb:.1f} MB)")

# Sanity check — T04 bundle compressed is ~120 MB; anything tiny is wrong.
if size_mb < 50:
    raise RuntimeError(
        f"Bundle zip is {size_mb:.1f} MB — far less than the expected ~120 MB. "
        f"The blob {BUNDLE_BLOB_NAME!r} is wrong, incomplete, or you fetched "
        f"the wrong project's bundle. Delete {local_zip} and verify the upload."
    )

with zipfile.ZipFile(local_zip) as zf:
    names = zf.namelist()
    has_azuredi = any(n.startswith("azuredi/") for n in names)
    has_pdfs    = any(n.startswith("pdfs/") for n in names)
    if not (has_azuredi and has_pdfs):
        sample = "\n  ".join(names[:10])
        raise RuntimeError(
            f"Zip {local_zip} does NOT contain the expected T04 layout "
            f"(azuredi/ and pdfs/ at top level). First entries:\n  {sample}"
        )
    zf.extractall(bundle_dir)

print("\nBundle contents:")
n_files = 0
total = 0
for p in sorted(bundle_dir.rglob("*")):
    if p.is_file() and p != local_zip:
        rel = p.relative_to(bundle_dir)
        size = p.stat().st_size
        total += size
        n_files += 1
        if n_files <= 20 or rel.suffix in (".csv", ".db"):
            print(f"  {str(rel):<55}  {size:>12,} B")
print(f"\n  ({n_files} files, {total/1024/1024:.1f} MB total)")


Bundle zip: /home/azureuser/bundle/v4_remaining4.zip (120.2 MB)

Bundle contents:
  azuredi/DocumentDefinitions.csv                                58,862 B
  azuredi/MyDocuments.csv                                         2,376 B
  azuredi/VectorDB_Documents.json                           316,245,443 B
  azuredi/VectorDBdev_Documents.json                          9,200,191 B
  dataset_creation_output/bsard_corpus.db                   102,551,552 B
  pdf_document_map.csv                                              165 B
  pdfs/1804_03_21_1804032150.pdf                              1,460,788 B
  pdfs/1867_06_08_1867060850.pdf                              1,645,048 B
  pdfs/1967_10_10_1967101055.pdf                              1,991,671 B
  pdfs/1967_10_10_1967101056.pdf                                858,705 B

  (10 files, 413.9 MB total)


*— cell 10 —*

## 5. Wire bundle paths into the T04 layout

`precompute_t04_indices.py` resolves paths via CLI flags. We construct a
`PATHS` dict here that's used by every downstream cell — so all inputs
come from the bundle, not the repo's own (empty on Azure) directories.

In [6]:
# === cell 11 ===
from pathlib import Path

bundle = Path(BUNDLE_DIR)
PATHS = {
    "azuredi_dir":  bundle / "azuredi",
    "pdf_doc_map":  bundle / "pdf_document_map.csv",
    "bsard_db":     bundle / "dataset_creation_output" / "bsard_corpus.db",
    "pdf_dir":      bundle / "pdfs",
    "cache_root":   Path(RESULTS_DIR),
}
PATHS["cache_root"].mkdir(parents=True, exist_ok=True)

required = ["azuredi_dir", "pdf_doc_map", "bsard_db", "pdf_dir"]


def _refresh_missing() -> list[str]:
    return [k for k in required if not PATHS[k].exists()]


missing = _refresh_missing()

# Auto-detect: if everything's missing and there's a single child dir that
# DOES contain `azuredi/`, the zip was packed with an extra wrapper level.
# Re-point PATHS one level deeper.
if len(missing) == len(required):
    children = [p for p in bundle.iterdir() if p.is_dir() and p.name != "__MACOSX"]
    candidates = [c for c in children if (c / "azuredi").exists()]
    if len(candidates) == 1:
        wrapped = candidates[0]
        print(f"[autodetect] bundle is wrapped under '{wrapped.name}/' — re-pointing PATHS.")
        PATHS["azuredi_dir"] = wrapped / "azuredi"
        PATHS["pdf_doc_map"] = wrapped / "pdf_document_map.csv"
        PATHS["bsard_db"]    = wrapped / "dataset_creation_output" / "bsard_corpus.db"
        PATHS["pdf_dir"]     = wrapped / "pdfs"
        missing = _refresh_missing()

if missing:
    print(f"\n[diagnostic] tree under {bundle}:")
    items = sorted(bundle.rglob("*"))
    if not items:
        print("  (empty — cell 9 may not have actually extracted anything)")
    else:
        for p in items[:40]:
            kind = "DIR " if p.is_dir() else "FILE"
            size = f"  ({p.stat().st_size:,} B)" if p.is_file() else ""
            print(f"  {kind}  {p.relative_to(bundle)}{size}")
        if len(items) > 40:
            print(f"  ... and {len(items) - 40} more")
    raise FileNotFoundError(
        f"Missing bundle inputs: {missing}. "
        "Expected top-level layout: azuredi/, dataset_creation_output/bsard_corpus.db, "
        "pdf_document_map.csv, pdfs/<stem>.pdf. "
        "Rebuild via `python scripts/prepare_azure_bundle.py` (writes a flat top-level zip) "
        "and re-upload."
    )

for stem in STEMS:
    pdf = PATHS["pdf_dir"] / f"{stem}.pdf"
    if not pdf.exists():
        raise FileNotFoundError(f"Bundle missing PDF: {pdf}")
    print(f"  {stem}.pdf  {pdf.stat().st_size/1024:.1f} KB")

print(f"\nResults will be written to: {PATHS['cache_root']}")


  1967_10_10_1967101056.pdf  838.6 KB
  1867_06_08_1867060850.pdf  1606.5 KB
  1804_03_21_1804032150.pdf  1426.6 KB
  1967_10_10_1967101055.pdf  1945.0 KB

Results will be written to: /home/azureuser/results


*— cell 12 —*

## 6. Idempotency helper — `is_stem_complete()`

Returns True when every (unit, variant) in `SMOKE_PLAN` for `stem` has
a config directory under `cache_root/<stem>/configs/` whose manifest
declares `linker_version == 4` AND whose four required files are
non-zero. Used by the smoke cell to short-circuit if doc 5 is already
built, and by each per-stem cell to skip the script entirely when the
stem is done.

In [12]:
# === cell 13 ===
import json
from pathlib import Path

REQUIRED_FILES = ["bm25.pkl", "faiss.index", "faiss_meta.json", "manifest.json"]

# Manifest schema (set by T04ConfigCache.write_manifest):
#   {
#     "schema_version": 1,
#     "doc_id": ...,
#     "config_hash": ...,
#     "chunking": {                  <-- chunking_params goes HERE, not "chunking_params"
#       "arm": "2a",
#       "variant": ...,
#       "unit": ...,
#       "linker_version": 4,
#       ...
#     },
#     ...
#   }
_MANIFEST_CHUNKING_KEY = "chunking"


def is_stem_complete(stem: str, cache_root: Path, smoke_plan=SMOKE_PLAN) -> bool:
    """True iff cache_root/<stem>/configs/ holds a valid config dir
    (linker_version=4, all 4 required files non-zero) for every (unit,
    variant) in smoke_plan."""
    cfg_dir = Path(cache_root) / stem / "configs"
    if not cfg_dir.exists():
        return False
    found_pairs = set()
    for sub in cfg_dir.iterdir():
        if not sub.is_dir():
            continue
        manifest = sub / "manifest.json"
        if not manifest.exists():
            continue
        try:
            m = json.loads(manifest.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            continue
        params = m.get(_MANIFEST_CHUNKING_KEY) or {}
        if params.get("linker_version") != 4 or params.get("arm") != "2a":
            continue
        # All required files present and non-zero?
        ok = all(
            (sub / f).exists() and (sub / f).stat().st_size > 0
            for f in REQUIRED_FILES
        )
        if not ok:
            continue
        found_pairs.add((params.get("unit"), params.get("variant")))
    needed = {tuple(p) for p in smoke_plan}
    return needed.issubset(found_pairs)


def stem_completion_status(stem: str, cache_root: Path, smoke_plan=SMOKE_PLAN) -> dict:
    """Diagnostic: which (unit, variant) pairs are built? Useful for
    spotting partial state without running the precompute."""
    cfg_dir = Path(cache_root) / stem / "configs"
    needed = {tuple(p) for p in smoke_plan}
    built = set()
    if cfg_dir.exists():
        for sub in cfg_dir.iterdir():
            if not sub.is_dir():
                continue
            manifest = sub / "manifest.json"
            if not manifest.exists():
                continue
            try:
                m = json.loads(manifest.read_text(encoding="utf-8"))
            except (json.JSONDecodeError, OSError):
                continue
            params = m.get(_MANIFEST_CHUNKING_KEY) or {}
            if params.get("linker_version") == 4 and params.get("arm") == "2a":
                ok = all((sub / f).exists() and (sub / f).stat().st_size > 0
                         for f in REQUIRED_FILES)
                if ok:
                    built.add((params.get("unit"), params.get("variant")))
    return {
        "built": sorted(built),
        "missing": sorted(needed - built),
        "n_built": len(built & needed),
        "n_needed": len(needed),
        "complete": needed.issubset(built),
    }


# Show current status for every target stem.
print("Current per-stem completion status:")
for s in STEMS:
    status = stem_completion_status(s, PATHS["cache_root"])
    print(f"  {s}: {status['n_built']}/{status['n_needed']} variants built"
          f"{' (COMPLETE)' if status['complete'] else ''}")


Current per-stem completion status:
  1967_10_10_1967101056: 6/6 variants built (COMPLETE)
  1867_06_08_1867060850: 0/6 variants built
  1804_03_21_1804032150: 0/6 variants built
  1967_10_10_1967101055: 0/6 variants built


*— cell 14 —*

## 7. Pre-flight (check_precompute_requirements.py)

Loads the AzureDI corpus, runs the BSARD linker, prints per-doc
coverage. Under linker v4 expect higher distinct-bsard_id counts than
v3 (doc 5: 308; doc 6: 410; doc 7: 772; doc 9: 434).

In [8]:
# === cell 15 ===
import subprocess
import sys

T04_REPO = repo_paths["T04_ARM2_METADATA"]
script = T04_REPO / "scripts" / "check_precompute_requirements.py"
cmd = [
    sys.executable, str(script),
    "--azuredi-dir", str(PATHS["azuredi_dir"]),
    "--pdf-doc-map", str(PATHS["pdf_doc_map"]),
    "--bsard-db",    str(PATHS["bsard_db"]),
    "--pdf-dir",     str(PATHS["pdf_dir"]),
    "--cache-root",  str(PATHS["cache_root"]),
]
print(" ".join(cmd))
r = subprocess.run(cmd)
if r.returncode != 0:
    raise RuntimeError("Preflight failed — see output above.")


/anaconda/envs/azureml_py38_PT_TF/bin/python /home/azureuser/repos/RQ2_T04_ARM2_METADATA/scripts/check_precompute_requirements.py --azuredi-dir /home/azureuser/bundle/azuredi --pdf-doc-map /home/azureuser/bundle/pdf_document_map.csv --bsard-db /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db --pdf-dir /home/azureuser/bundle/pdfs --cache-root /home/azureuser/results
  T04 precompute — preflight requirements check

1. Python interpreter
---------------------
  [OK]   Python 3.10.0 (>= 3.10)
  [OK]   executable: /anaconda/envs/azureml_py38_PT_TF/bin/python

2. Required packages
--------------------
  [OK]   torch 2.9.1+cu128
  [OK]   sentence_transformers 5.3.0
  [OK]   transformers 5.4.0
  [OK]   faiss 1.13.2
  [OK]   rank_bm25 ?
  [OK]   fitz 1.27.2.3
  [OK]   pandas 2.3.3
  [OK]   numpy 1.26.4
  [OK]   scipy 1.15.3
  [OK]   ollama ?

2b. Sibling editable installs
-----------------------------
  [OK]   shared -> /home/azureuser/repos/RQ2_T01_SHARED/src/shared
  [OK]   data

*— cell 16 —*

## 8. Smoke — full SMOKE_PLAN build on doc 5

Builds **all 6 SMOKE_PLAN variants on the smallest stem (doc 5)** and
times each one independently via a single `precompute_t04_indices.py`
call. The per-variant wall times are extracted from the script's log
and persisted to `<cache_root>/.smoke_timings.json` so the time-gate
cell below survives kernel restarts.

If doc 5 is already complete (this notebook was re-run after a kernel
restart), the smoke cell loads timings from
`<cache_root>/.smoke_timings.json` instead and only re-runs the script
when the timings file is missing.

Expected wall on first run (fresh build): ~6 min on T4, ~3 min on A100.
Cache-hit re-run: ~10 seconds.

In [9]:
# === cell 17 ===
import json
import re
import subprocess
import sys
import time
from pathlib import Path

timings_path = PATHS["cache_root"] / SMOKE_TIMINGS_FILE
smoke_complete = is_stem_complete(SMOKE_STEM, PATHS["cache_root"])
timings: dict[str, float] = {}

if smoke_complete and timings_path.exists():
    timings = json.loads(timings_path.read_text(encoding="utf-8"))
    print(f"[skip] {SMOKE_STEM} already complete; loaded timings from {timings_path}")
else:
    if smoke_complete and not timings_path.exists():
        print(f"[note] {SMOKE_STEM} already complete but no timings file — "
              "re-running smoke for measurements (variants will [cache hit] fast).")
    cmd = [
        sys.executable, str(T04_REPO / "scripts" / "precompute_t04_indices.py"),
        "--doc-id",          SMOKE_STEM,
        "--pdf-dir",         str(PATHS["pdf_dir"]),
        "--azuredi-dir",     str(PATHS["azuredi_dir"]),
        "--pdf-doc-map",     str(PATHS["pdf_doc_map"]),
        "--bsard-db",        str(PATHS["bsard_db"]),
        "--cache-root",      str(PATHS["cache_root"]),
        "--embedding-model", EMBEDDING_MODEL,
        "-v",
    ]
    print(" ".join(cmd))
    t0 = time.perf_counter()
    r = subprocess.run(cmd, capture_output=True, text=True)
    wall = time.perf_counter() - t0
    print(r.stdout[-4000:] if r.stdout else "")
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("Smoke build failed.")

    # Parse per-variant timings from the script's log.
    # Match: "[built]    node/raw -> 261f1a1cd37b   n_units=619  truncated=12 (1.9%)  in 53.2s"
    pat = re.compile(
        r"\[built\]\s+(?P<unit>node|article)/(?P<variant>\w+)\s+->\s+\S+.*?in\s+(?P<sec>[\d.]+)s",
        re.IGNORECASE,
    )
    log = (r.stdout or "") + "\n" + (r.stderr or "")
    for m in pat.finditer(log):
        key = f"{m.group('unit')}/{m.group('variant')}"
        timings[key] = float(m.group("sec"))

    # Anything we didn't find in the log probably cache-hit (already on disk
    # before this cell ran). Record 0.0 for those so the time-gate can warn.
    for unit, variant in SMOKE_PLAN:
        key = f"{unit}/{variant}"
        timings.setdefault(key, 0.0)

    timings_path.parent.mkdir(parents=True, exist_ok=True)
    timings_path.write_text(json.dumps(timings, indent=2), encoding="utf-8")
    print(f"\nSmoke wall: {wall:.1f}s (sum-of-variants {sum(timings.values()):.1f}s)")
    print(f"Persisted per-variant timings -> {timings_path}")

print("\nPer-variant times on", SMOKE_STEM, ":")
for unit, variant in SMOKE_PLAN:
    key = f"{unit}/{variant}"
    s = timings.get(key, 0.0)
    marker = " (cache-hit — not representative)" if s < 1.0 else ""
    print(f"  {key:<20s}  {s:>7.1f} s{marker}")


/anaconda/envs/azureml_py38_PT_TF/bin/python /home/azureuser/repos/RQ2_T04_ARM2_METADATA/scripts/precompute_t04_indices.py --doc-id 1967_10_10_1967101056 --pdf-dir /home/azureuser/bundle/pdfs --azuredi-dir /home/azureuser/bundle/azuredi --pdf-doc-map /home/azureuser/bundle/pdf_document_map.csv --bsard-db /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db --cache-root /home/azureuser/results --embedding-model intfloat/multilingual-e5-large-instruct -v
plan (6):                 [('node', 'raw'), ('node', 'enriched'), ('node', 'summary'), ('node', 'full'), ('article', 'raw'), ('article', 'full')]
doc_id (BSARD stem):          1967_10_10_1967101056
azure_document_id:            5
embedding model:              intfloat/multilingual-e5-large-instruct
tokenizer:                    intfloat/multilingual-e5-large-instruct
sparse_only:                  False
force:                        False
azuredi:                      /home/azureuser/bundle/azuredi
bsard_db:                     

*— cell 18 —*

## 9. Time-budget gate

Uses the per-variant wall times measured on doc 5 to project the other
3 stems. Per-variant per-unit rate × per-stem unit count = projected
wall. Inspect the projection below; **only run the per-stem cells
once the numbers look acceptable.**

In [10]:
# === cell 19 ===
n_nodes_doc5, n_articles_doc5 = WORKLOAD[SMOKE_STEM]

per_unit_s: dict[str, float] = {}
for unit, variant in SMOKE_PLAN:
    key = f"{unit}/{variant}"
    units_doc5 = n_nodes_doc5 if unit == "node" else n_articles_doc5
    per_unit_s[key] = timings.get(key, 0.0) / max(units_doc5, 1)

print("Per-variant per-unit rates (s/unit, from doc 5 smoke):")
for k, v in per_unit_s.items():
    print(f"  {k:<20s}  {v*1000:>7.1f} ms/unit")

print()
print("Projected per-stem wall (sum across 6 variants):")
total_remaining = 0.0
for s in STEMS:
    n_n, n_a = WORKLOAD[s]
    stem_total = 0.0
    for unit, variant in SMOKE_PLAN:
        key = f"{unit}/{variant}"
        units = n_n if unit == "node" else n_a
        stem_total += per_unit_s[key] * units
    if s == SMOKE_STEM:
        marker = "  [done in smoke]"
    else:
        total_remaining += stem_total
        marker = ""
    print(f"  {s}:  ~{stem_total/60:>6.1f} min  ({stem_total:>6.0f}s){marker}")

print(f"\nProjected REMAINING wall (doc 6 + doc 9 + doc 7): ~{total_remaining/60:.1f} min")
print(f"                                                       ({total_remaining/3600:.2f} h)")
print()
print("Inspect — then run the per-stem cells below.")


Per-variant per-unit rates (s/unit, from doc 5 smoke):
  node/raw                  9.5 ms/unit
  node/enriched             6.3 ms/unit
  node/summary             11.1 ms/unit
  node/full                 6.3 ms/unit
  article/raw               8.2 ms/unit
  article/full              8.2 ms/unit

Projected per-stem wall (sum across 6 variants):
  1967_10_10_1967101056:  ~   0.4 min  (    25s)  [done in smoke]
  1867_06_08_1867060850:  ~   0.5 min  (    29s)
  1804_03_21_1804032150:  ~   0.6 min  (    39s)
  1967_10_10_1967101055:  ~   1.1 min  (    66s)

Projected REMAINING wall (doc 6 + doc 9 + doc 7): ~2.2 min
                                                       (0.04 h)

Inspect — then run the per-stem cells below.


*— cell 20 —*

## 10. Per-stem precompute (4 cells, idempotent)

Each cell runs `precompute_t04_indices.py` for one stem. **Each cell
short-circuits with `[skip]` if the stem is already fully built under
linker v4** (no script invocation, no model load) — so re-running the
notebook after a kernel restart safely skips finished stems.

The doc 5 cell will skip immediately on a normal run because the smoke
cell above already built it. It's kept for symmetry / explicit "doc 5
is done" reporting.

In [13]:
# === cell 21 ===
# Doc 5 — Code Judiciaire (smaller)  — est ~6 min on T4, ~4 min on A100
import subprocess
import sys
import time

STEM = "1967_10_10_1967101056"
status = stem_completion_status(STEM, PATHS["cache_root"])

if status["complete"]:
    print(f"[skip] {STEM}: all {status['n_needed']}/{status['n_needed']} "
          "SMOKE_PLAN configs already built under linker v4 "
          f"(at {PATHS['cache_root'] / STEM / 'configs'})")
else:
    if status["n_built"] > 0:
        print(f"[partial] {STEM}: {status['n_built']}/{status['n_needed']} "
              f"variants already built; will rebuild only missing {status['missing']}")
    cmd = [
        sys.executable, str(repo_paths["T04_ARM2_METADATA"] / "scripts" / "precompute_t04_indices.py"),
        "--doc-id",          STEM,
        "--pdf-dir",         str(PATHS["pdf_dir"]),
        "--azuredi-dir",     str(PATHS["azuredi_dir"]),
        "--pdf-doc-map",     str(PATHS["pdf_doc_map"]),
        "--bsard-db",        str(PATHS["bsard_db"]),
        "--cache-root",      str(PATHS["cache_root"]),
        "--embedding-model", EMBEDDING_MODEL,
        "-v",
    ]
    print(" ".join(cmd))
    t0 = time.perf_counter()
    r = subprocess.run(cmd)
    dt = time.perf_counter() - t0
    print(f"\n=== {STEM} done: wall {dt/60:.1f} min  exit={r.returncode} ===")
    if r.returncode != 0:
        raise RuntimeError(f"precompute failed on {STEM}.")
    # Post-run sanity: stem must now report complete.
    if not is_stem_complete(STEM, PATHS["cache_root"]):
        post = stem_completion_status(STEM, PATHS["cache_root"])
        raise RuntimeError(
            f"{STEM} ran but is_stem_complete=False; missing={post['missing']}"
        )


[skip] 1967_10_10_1967101056: all 6/6 SMOKE_PLAN configs already built under linker v4 (at /home/azureuser/results/1967_10_10_1967101056/configs)


In [14]:
# === cell 22 ===
# Doc 6 — Code Pénal  — est ~7 min on T4, ~5 min on A100
import subprocess
import sys
import time

STEM = "1867_06_08_1867060850"
status = stem_completion_status(STEM, PATHS["cache_root"])

if status["complete"]:
    print(f"[skip] {STEM}: all {status['n_needed']}/{status['n_needed']} "
          "SMOKE_PLAN configs already built under linker v4 "
          f"(at {PATHS['cache_root'] / STEM / 'configs'})")
else:
    if status["n_built"] > 0:
        print(f"[partial] {STEM}: {status['n_built']}/{status['n_needed']} "
              f"variants already built; will rebuild only missing {status['missing']}")
    cmd = [
        sys.executable, str(repo_paths["T04_ARM2_METADATA"] / "scripts" / "precompute_t04_indices.py"),
        "--doc-id",          STEM,
        "--pdf-dir",         str(PATHS["pdf_dir"]),
        "--azuredi-dir",     str(PATHS["azuredi_dir"]),
        "--pdf-doc-map",     str(PATHS["pdf_doc_map"]),
        "--bsard-db",        str(PATHS["bsard_db"]),
        "--cache-root",      str(PATHS["cache_root"]),
        "--embedding-model", EMBEDDING_MODEL,
        "-v",
    ]
    print(" ".join(cmd))
    t0 = time.perf_counter()
    r = subprocess.run(cmd)
    dt = time.perf_counter() - t0
    print(f"\n=== {STEM} done: wall {dt/60:.1f} min  exit={r.returncode} ===")
    if r.returncode != 0:
        raise RuntimeError(f"precompute failed on {STEM}.")
    # Post-run sanity: stem must now report complete.
    if not is_stem_complete(STEM, PATHS["cache_root"]):
        post = stem_completion_status(STEM, PATHS["cache_root"])
        raise RuntimeError(
            f"{STEM} ran but is_stem_complete=False; missing={post['missing']}"
        )


/anaconda/envs/azureml_py38_PT_TF/bin/python /home/azureuser/repos/RQ2_T04_ARM2_METADATA/scripts/precompute_t04_indices.py --doc-id 1867_06_08_1867060850 --pdf-dir /home/azureuser/bundle/pdfs --azuredi-dir /home/azureuser/bundle/azuredi --pdf-doc-map /home/azureuser/bundle/pdf_document_map.csv --bsard-db /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db --cache-root /home/azureuser/results --embedding-model intfloat/multilingual-e5-large-instruct -v


10:26:19 [INFO] arm2_metadata.azuredi_loader: Kept document_ids from MyDocuments.csv: [5, 6, 7, 8, 9]
10:27:04 [INFO] arm2_metadata.azuredi_loader: Loaded 5749 AzureDI nodes (per doc: {5: 668, 6: 858, 8: 1309, 9: 1115, 7: 1799}); skipped 871 records from non-kept doc_ids: {2: 615, 1: 33, 3: 18, 4: 18, 45: 60, 46: 127}
10:27:04  INFO      shared.embeddings — Loading embedding model: intfloat/multilingual-e5-large-instruct
10:27:04 [INFO] shared.embeddings: Loading embedding model: intfloat/multilingual-e5-large-instruct
10:27:04 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: cuda:0
10:27:04 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: intfloat/multilingual-e5-large-instruct
10:27:04 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
10:27:04 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/

plan (6):                 [('node', 'raw'), ('node', 'enriched'), ('node', 'summary'), ('node', 'full'), ('article', 'raw'), ('article', 'full')]
doc_id (BSARD stem):          1867_06_08_1867060850
azure_document_id:            6
embedding model:              intfloat/multilingual-e5-large-instruct
tokenizer:                    intfloat/multilingual-e5-large-instruct
sparse_only:                  False
force:                        False
azuredi:                      /home/azureuser/bundle/azuredi
bsard_db:                     /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db
pdf_path:                     /home/azureuser/bundle/pdfs/1867_06_08_1867060850.pdf (OK)
cache_root:                   /home/azureuser/results

indexable nodes:              680
indexable articles:           397

Loading embedding model …  (CUDA: available — device 0 = Tesla T4)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5511.83it/s]
10:27:06 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
10:27:06 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large-instruct/274baa43b0e13e37fafa6428dbc7938e62e5c439/config.json "HTTP/1.1 200 OK"
10:27:06 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
10:27:06 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large-instruct/274baa43b0e13e37fafa6428dbc7938e62e5c439/tokenizer_config.json "HTTP/1.1 200 OK"
10:27:06 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/intfloat/multilingual-e5-large-instruct/tree/main/additional_chat_templates?recursive=false&expand=false "


azuredi_fingerprint:          b8b1efae4e866e6a5f04cb88…
pdf_sha256:                   f727868096ffad850a807035…

Building 6 (unit, variant) configs …


Batches: 100%|██████████| 11/11 [00:04<00:00,  2.67it/s]
10:27:15  INFO      shared.bm25_store — BM25Store built with 680 documents (k1=1.500, b=0.250)
10:27:15 [INFO] shared.bm25_store: BM25Store built with 680 documents (k1=1.500, b=0.250)
10:27:15  INFO      shared.faiss_store — FAISSStore saved to /home/azureuser/results/1867_06_08_1867060850/configs/c930b0090cb5 (680 vectors)
10:27:15 [INFO] shared.faiss_store: FAISSStore saved to /home/azureuser/results/1867_06_08_1867060850/configs/c930b0090cb5 (680 vectors)
10:27:15  INFO      shared.bm25_store — BM25Store saved to /home/azureuser/results/1867_06_08_1867060850/configs/c930b0090cb5/bm25.pkl (680 documents, k1=1.500, b=0.250)
10:27:15 [INFO] shared.bm25_store: BM25Store saved to /home/azureuser/results/1867_06_08_1867060850/configs/c930b0090cb5/bm25.pkl (680 documents, k1=1.500, b=0.250)
10:27:15 [INFO] __main__: [built]    node/raw -> c930b0090cb5   n_units=680  truncated=22 (3.2%)  in 4.5s
10:27:15 [INFO] __main__: [building] n


Done.
  unit     variant    config_hash    status  
  node     raw        c930b0090cb5   built   
  node     enriched   632ccfc904d9   built   
  node     summary    a5e58fa64547   built   
  node     full       722eb25059b0   built   
  article  raw        0cae06c07782   built   
  article  full       71bb466decf0   built   

=== 1867_06_08_1867060850 done: wall 1.4 min  exit=0 ===


In [15]:
# === cell 23 ===
# Doc 9 — Code Civil  — est ~10 min on T4, ~6 min on A100
import subprocess
import sys
import time

STEM = "1804_03_21_1804032150"
status = stem_completion_status(STEM, PATHS["cache_root"])

if status["complete"]:
    print(f"[skip] {STEM}: all {status['n_needed']}/{status['n_needed']} "
          "SMOKE_PLAN configs already built under linker v4 "
          f"(at {PATHS['cache_root'] / STEM / 'configs'})")
else:
    if status["n_built"] > 0:
        print(f"[partial] {STEM}: {status['n_built']}/{status['n_needed']} "
              f"variants already built; will rebuild only missing {status['missing']}")
    cmd = [
        sys.executable, str(repo_paths["T04_ARM2_METADATA"] / "scripts" / "precompute_t04_indices.py"),
        "--doc-id",          STEM,
        "--pdf-dir",         str(PATHS["pdf_dir"]),
        "--azuredi-dir",     str(PATHS["azuredi_dir"]),
        "--pdf-doc-map",     str(PATHS["pdf_doc_map"]),
        "--bsard-db",        str(PATHS["bsard_db"]),
        "--cache-root",      str(PATHS["cache_root"]),
        "--embedding-model", EMBEDDING_MODEL,
        "-v",
    ]
    print(" ".join(cmd))
    t0 = time.perf_counter()
    r = subprocess.run(cmd)
    dt = time.perf_counter() - t0
    print(f"\n=== {STEM} done: wall {dt/60:.1f} min  exit={r.returncode} ===")
    if r.returncode != 0:
        raise RuntimeError(f"precompute failed on {STEM}.")
    # Post-run sanity: stem must now report complete.
    if not is_stem_complete(STEM, PATHS["cache_root"]):
        post = stem_completion_status(STEM, PATHS["cache_root"])
        raise RuntimeError(
            f"{STEM} ran but is_stem_complete=False; missing={post['missing']}"
        )


/anaconda/envs/azureml_py38_PT_TF/bin/python /home/azureuser/repos/RQ2_T04_ARM2_METADATA/scripts/precompute_t04_indices.py --doc-id 1804_03_21_1804032150 --pdf-dir /home/azureuser/bundle/pdfs --azuredi-dir /home/azureuser/bundle/azuredi --pdf-doc-map /home/azureuser/bundle/pdf_document_map.csv --bsard-db /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db --cache-root /home/azureuser/results --embedding-model intfloat/multilingual-e5-large-instruct -v


10:28:32 [INFO] arm2_metadata.azuredi_loader: Kept document_ids from MyDocuments.csv: [5, 6, 7, 8, 9]
10:29:19 [INFO] arm2_metadata.azuredi_loader: Loaded 5749 AzureDI nodes (per doc: {5: 668, 6: 858, 8: 1309, 9: 1115, 7: 1799}); skipped 871 records from non-kept doc_ids: {2: 615, 1: 33, 3: 18, 4: 18, 45: 60, 46: 127}
10:29:19  INFO      shared.embeddings — Loading embedding model: intfloat/multilingual-e5-large-instruct
10:29:19 [INFO] shared.embeddings: Loading embedding model: intfloat/multilingual-e5-large-instruct
10:29:19 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: cuda:0
10:29:19 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: intfloat/multilingual-e5-large-instruct
10:29:19 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
10:29:19 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenti

plan (6):                 [('node', 'raw'), ('node', 'enriched'), ('node', 'summary'), ('node', 'full'), ('article', 'raw'), ('article', 'full')]
doc_id (BSARD stem):          1804_03_21_1804032150
azure_document_id:            9
embedding model:              intfloat/multilingual-e5-large-instruct
tokenizer:                    intfloat/multilingual-e5-large-instruct
sparse_only:                  False
force:                        False
azuredi:                      /home/azureuser/bundle/azuredi
bsard_db:                     /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db
pdf_path:                     /home/azureuser/bundle/pdfs/1804_03_21_1804032150.pdf (OK)
cache_root:                   /home/azureuser/results

indexable nodes:              958
indexable articles:           429

Loading embedding model …  (CUDA: available — device 0 = Tesla T4)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9922.39it/s]
10:29:21 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
10:29:21 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large-instruct/274baa43b0e13e37fafa6428dbc7938e62e5c439/config.json "HTTP/1.1 200 OK"
10:29:21 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
10:29:21 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large-instruct/274baa43b0e13e37fafa6428dbc7938e62e5c439/tokenizer_config.json "HTTP/1.1 200 OK"
10:29:21 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/intfloat/multilingual-e5-large-instruct/tree/main/additional_chat_templates?recursive=false&expand=false "


azuredi_fingerprint:          b8b1efae4e866e6a5f04cb88…
pdf_sha256:                   e36e0674a64d0fc7fc8dbb2e…

Building 6 (unit, variant) configs …


Batches: 100%|██████████| 15/15 [00:04<00:00,  3.14it/s]
10:29:32  INFO      shared.bm25_store — BM25Store built with 958 documents (k1=1.500, b=0.250)
10:29:32 [INFO] shared.bm25_store: BM25Store built with 958 documents (k1=1.500, b=0.250)
10:29:32  INFO      shared.faiss_store — FAISSStore saved to /home/azureuser/results/1804_03_21_1804032150/configs/5aa9d3f0e4d5 (958 vectors)
10:29:32 [INFO] shared.faiss_store: FAISSStore saved to /home/azureuser/results/1804_03_21_1804032150/configs/5aa9d3f0e4d5 (958 vectors)
10:29:32  INFO      shared.bm25_store — BM25Store saved to /home/azureuser/results/1804_03_21_1804032150/configs/5aa9d3f0e4d5/bm25.pkl (958 documents, k1=1.500, b=0.250)
10:29:32 [INFO] shared.bm25_store: BM25Store saved to /home/azureuser/results/1804_03_21_1804032150/configs/5aa9d3f0e4d5/bm25.pkl (958 documents, k1=1.500, b=0.250)
10:29:32 [INFO] __main__: [built]    node/raw -> 5aa9d3f0e4d5   n_units=958  truncated=26 (2.7%)  in 5.2s
10:29:32 [INFO] __main__: [building] n


Done.
  unit     variant    config_hash    status  
  node     raw        5aa9d3f0e4d5   built   
  node     enriched   c43a26fb572f   built   
  node     summary    e4167c4f052e   built   
  node     full       0a189183e751   built   
  article  raw        45ef6113e0fd   built   
  article  full       dd935503ddd6   built   

=== 1804_03_21_1804032150 done: wall 1.5 min  exit=0 ===


In [16]:
# === cell 24 ===
# Doc 7 — Code Judiciaire (larger)  — est ~17 min on T4, ~11 min on A100
import subprocess
import sys
import time

STEM = "1967_10_10_1967101055"
status = stem_completion_status(STEM, PATHS["cache_root"])

if status["complete"]:
    print(f"[skip] {STEM}: all {status['n_needed']}/{status['n_needed']} "
          "SMOKE_PLAN configs already built under linker v4 "
          f"(at {PATHS['cache_root'] / STEM / 'configs'})")
else:
    if status["n_built"] > 0:
        print(f"[partial] {STEM}: {status['n_built']}/{status['n_needed']} "
              f"variants already built; will rebuild only missing {status['missing']}")
    cmd = [
        sys.executable, str(repo_paths["T04_ARM2_METADATA"] / "scripts" / "precompute_t04_indices.py"),
        "--doc-id",          STEM,
        "--pdf-dir",         str(PATHS["pdf_dir"]),
        "--azuredi-dir",     str(PATHS["azuredi_dir"]),
        "--pdf-doc-map",     str(PATHS["pdf_doc_map"]),
        "--bsard-db",        str(PATHS["bsard_db"]),
        "--cache-root",      str(PATHS["cache_root"]),
        "--embedding-model", EMBEDDING_MODEL,
        "-v",
    ]
    print(" ".join(cmd))
    t0 = time.perf_counter()
    r = subprocess.run(cmd)
    dt = time.perf_counter() - t0
    print(f"\n=== {STEM} done: wall {dt/60:.1f} min  exit={r.returncode} ===")
    if r.returncode != 0:
        raise RuntimeError(f"precompute failed on {STEM}.")
    # Post-run sanity: stem must now report complete.
    if not is_stem_complete(STEM, PATHS["cache_root"]):
        post = stem_completion_status(STEM, PATHS["cache_root"])
        raise RuntimeError(
            f"{STEM} ran but is_stem_complete=False; missing={post['missing']}"
        )


/anaconda/envs/azureml_py38_PT_TF/bin/python /home/azureuser/repos/RQ2_T04_ARM2_METADATA/scripts/precompute_t04_indices.py --doc-id 1967_10_10_1967101055 --pdf-dir /home/azureuser/bundle/pdfs --azuredi-dir /home/azureuser/bundle/azuredi --pdf-doc-map /home/azureuser/bundle/pdf_document_map.csv --bsard-db /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db --cache-root /home/azureuser/results --embedding-model intfloat/multilingual-e5-large-instruct -v


10:30:13 [INFO] arm2_metadata.azuredi_loader: Kept document_ids from MyDocuments.csv: [5, 6, 7, 8, 9]
10:30:59 [INFO] arm2_metadata.azuredi_loader: Loaded 5749 AzureDI nodes (per doc: {5: 668, 6: 858, 8: 1309, 9: 1115, 7: 1799}); skipped 871 records from non-kept doc_ids: {2: 615, 1: 33, 3: 18, 4: 18, 45: 60, 46: 127}
10:30:59  INFO      shared.embeddings — Loading embedding model: intfloat/multilingual-e5-large-instruct
10:30:59 [INFO] shared.embeddings: Loading embedding model: intfloat/multilingual-e5-large-instruct
10:30:59 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: cuda:0
10:30:59 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: intfloat/multilingual-e5-large-instruct
10:30:59 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
10:30:59 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/

plan (6):                 [('node', 'raw'), ('node', 'enriched'), ('node', 'summary'), ('node', 'full'), ('article', 'raw'), ('article', 'full')]
doc_id (BSARD stem):          1967_10_10_1967101055
azure_document_id:            7
embedding model:              intfloat/multilingual-e5-large-instruct
tokenizer:                    intfloat/multilingual-e5-large-instruct
sparse_only:                  False
force:                        False
azuredi:                      /home/azureuser/bundle/azuredi
bsard_db:                     /home/azureuser/bundle/dataset_creation_output/bsard_corpus.db
pdf_path:                     /home/azureuser/bundle/pdfs/1967_10_10_1967101055.pdf (OK)
cache_root:                   /home/azureuser/results

indexable nodes:              1616
indexable articles:           767

Loading embedding model …  (CUDA: available — device 0 = Tesla T4)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5576.68it/s]
10:31:01 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
10:31:01 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large-instruct/274baa43b0e13e37fafa6428dbc7938e62e5c439/config.json "HTTP/1.1 200 OK"
10:31:01 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large-instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
10:31:01 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large-instruct/274baa43b0e13e37fafa6428dbc7938e62e5c439/tokenizer_config.json "HTTP/1.1 200 OK"
10:31:01 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/intfloat/multilingual-e5-large-instruct/tree/main/additional_chat_templates?recursive=false&expand=false "


azuredi_fingerprint:          b8b1efae4e866e6a5f04cb88…
pdf_sha256:                   7e8ff1297b7e5574df496d34…

Building 6 (unit, variant) configs …


Batches: 100%|██████████| 26/26 [00:07<00:00,  3.69it/s]
10:31:13  INFO      shared.bm25_store — BM25Store built with 1616 documents (k1=1.500, b=0.250)
10:31:13 [INFO] shared.bm25_store: BM25Store built with 1616 documents (k1=1.500, b=0.250)
10:31:14  INFO      shared.faiss_store — FAISSStore saved to /home/azureuser/results/1967_10_10_1967101055/configs/4e8826f4d25c (1616 vectors)
10:31:14 [INFO] shared.faiss_store: FAISSStore saved to /home/azureuser/results/1967_10_10_1967101055/configs/4e8826f4d25c (1616 vectors)
10:31:14  INFO      shared.bm25_store — BM25Store saved to /home/azureuser/results/1967_10_10_1967101055/configs/4e8826f4d25c/bm25.pkl (1616 documents, k1=1.500, b=0.250)
10:31:14 [INFO] shared.bm25_store: BM25Store saved to /home/azureuser/results/1967_10_10_1967101055/configs/4e8826f4d25c/bm25.pkl (1616 documents, k1=1.500, b=0.250)
10:31:14 [INFO] __main__: [built]    node/raw -> 4e8826f4d25c   n_units=1616  truncated=29 (1.8%)  in 7.7s
10:31:14 [INFO] __main__: [buil


Done.
  unit     variant    config_hash    status  
  node     raw        4e8826f4d25c   built   
  node     enriched   0e245975c315   built   
  node     summary    45f35213bdc3   built   
  node     full       fbd789707e0c   built   
  article  raw        b09caeaf1a41   built   
  article  full       b8f7aff547f3   built   

=== 1967_10_10_1967101055 done: wall 1.8 min  exit=0 ===


*— cell 25 —*

## 11. Verify all 4 stems are now complete

Cross-checks every stem via `is_stem_complete()`. Fails loudly if any
stem is short of the 6 SMOKE_PLAN configs.

In [17]:
# === cell 26 ===
all_good = True
for stem in STEMS:
    status = stem_completion_status(stem, PATHS["cache_root"])
    if status["complete"]:
        cfg_dir = PATHS["cache_root"] / stem / "configs"
        dirs = [d for d in cfg_dir.iterdir() if d.is_dir()]
        print(f"  [{stem}]  OK  ({len(dirs)} dirs)")
    else:
        print(f"  [{stem}]  INCOMPLETE  built={status['n_built']}/{status['n_needed']}  missing={status['missing']}")
        all_good = False

if not all_good:
    raise RuntimeError("One or more stems are incomplete — investigate before upload.")
print("\nAll 4 stems verified complete under linker v4.")


  [1967_10_10_1967101056]  OK  (6 dirs)
  [1867_06_08_1867060850]  OK  (6 dirs)
  [1804_03_21_1804032150]  OK  (6 dirs)
  [1967_10_10_1967101055]  OK  (6 dirs)

All 4 stems verified complete under linker v4.


*— cell 27 —*

## 12. Upload results to Azure Blob

Mirrors the layout `<stem>/configs/<v4-hash>/<file>` under
`RESULTS_BLOB_PREFIX` so a single `azcopy sync` from the local
results target picks up the whole tree.

In [18]:
# === cell 28 ===
n_uploaded = 0
total_bytes = 0
for stem in STEMS:
    cfg_dir = PATHS["cache_root"] / stem / "configs"
    for sub in cfg_dir.iterdir():
        if not sub.is_dir():
            continue
        for path in sub.iterdir():
            if not path.is_file():
                continue
            blob_name = f"{RESULTS_BLOB_PREFIX}/{stem}/configs/{sub.name}/{path.name}"
            with path.open("rb") as f:
                container.get_blob_client(blob_name).upload_blob(f, overwrite=True)
            n_uploaded += 1
            total_bytes += path.stat().st_size

print(f"Uploaded {n_uploaded} files ({total_bytes/1024/1024:.1f} MB) under {RESULTS_BLOB_PREFIX}/")


Uploaded 120 files (132.3 MB) under t04_results/v4_remaining4/


*— cell 29 —*

## 13. Stage results for `scp` back to the local results dir

Builds a directory tree on the VM that mirrors the local
`RQ2_T04_ARM2_METADATA/data/<stem>/configs/<v4-hash>/` layout, so a
single `scp -r` drops the new configs straight into the results dir.


In [19]:
# === cell 30 ===
import shutil
from pathlib import Path

EXPORT_ROOT = Path("/home/azureuser/local_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True)

n_files = 0
for stem in STEMS:
    src = PATHS["cache_root"] / stem / "configs"
    dst = EXPORT_ROOT / stem / "configs"
    dst.mkdir(parents=True)
    for sub in src.iterdir():
        if not sub.is_dir():
            continue
        target = dst / sub.name
        shutil.copytree(sub, target)
        n_files += sum(1 for _ in target.rglob("*") if _.is_file())

print(f"Staged {n_files} files at {EXPORT_ROOT}")
print()
print("To copy them down (replace <vm-ip> and <repo-root>):")
print(f"  scp -r azureuser@<vm-ip>:{EXPORT_ROOT}/. \\")
print(f'         "<repo-root>/RQ2_T04_ARM2_METADATA/data/"')


Staged 120 files at /home/azureuser/local_export

To copy them down (replace <vm-ip> and <repo-root>):
  scp -r azureuser@<vm-ip>:/home/azureuser/local_export/. \
         "<repo-root>/RQ2_T04_ARM2_METADATA/data/"


*— cell 31 —*

## 14. Cleanup

Free CUDA memory and shut down. No background daemon to kill (T04
doesn't use Ollama, unlike T05).

In [20]:
# === cell 32 ===
import gc
import importlib

gc.collect()
try:
    torch = importlib.import_module("torch")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA cache cleared.")
except ImportError:
    pass
print("Done.")


CUDA cache cleared.
Done.
